In [1]:
# TODO check the intersection between croplands from SPAM and MODIS land classes.
# It's possible that there will be gaps to fill.

# Creating an inventory of feed (energy, protein, NDF)

- We have CSVs for each production system, each of which has 832,827 rows, and columns for point in time production values for each crop. - - These data points can be spatially mapped. 
- Our first task will be to turn these CSVs into raster files for each crop.
- We'll then use those raster values to get dry crop residue mass, energy, protein and NDF, all on a per-pixel basis

### Translate CSVs into raster files

In [1]:
import pandas as pd

IRRIGATED_PROD_CSV = "../../data/processed/SPAM/production_pit_TI.csv"
RAINFED_HIGH_INPUTS_PROD_CSV = "../../data/processed/SPAM/production_pit_TH.csv"
RAINFED_LOW_INPUTS_PROD_CSV = "../../data/processed/SPAM/production_pit_TL.csv"
SUBSISTENCE_PROD_CSV = "../../data/processed/SPAM/production_pit_TS.csv"
SPAM_CROP_LIST_CSV = "../../data/external/spam_crop_list2.csv"

In [2]:
# Function to rename columns in a DataFrame based on the provided mapping
def rename_columns(df, column_mapping):
    # Create a new dictionary for the column mapping with the suffixes
    new_column_mapping = {f"{short}_{suffix}": f"{long}_pit" for short, long in column_mapping.items() for suffix in ['i', 'h', 'l', 's']}
    
    # Update column names
    df.rename(columns=new_column_mapping, inplace=True)
    return df





In [3]:
spam_croplist = pd.read_csv(SPAM_CROP_LIST_CSV)

# Create a dictionary for mapping short names to long names
spam_crop_mapping = dict(zip(spam_croplist['spam_short'], spam_croplist['spam_long']))

i_df = pd.read_csv(IRRIGATED_PROD_CSV)
h_df = pd.read_csv(RAINFED_HIGH_INPUTS_PROD_CSV)
l_df = pd.read_csv(RAINFED_LOW_INPUTS_PROD_CSV)
s_df = pd.read_csv(SUBSISTENCE_PROD_CSV)

# Rename columns for each DataFrame
i_df = rename_columns(i_df, spam_crop_mapping)
h_df = rename_columns(h_df, spam_crop_mapping)
l_df = rename_columns(l_df, spam_crop_mapping)
s_df = rename_columns(s_df, spam_crop_mapping)

In [4]:
i_df.columns

Index(['iso3', 'prod_level', 'alloc_key', 'cell5m', 'x', 'y', 'rec_type',
       'tech_type', 'unit', 'wheat_pit', 'rice_pit', 'maize_pit', 'barley_pit',
       'pearlmill_pit', 'smallmill_pit', 'sorghum_pit', 'oth_cereal_pit',
       'potato_pit', 'sweet_pot_pit', 'yams_pit', 'cassava_pit',
       'oth_root_pit', 'bean_pit', 'chickpea_pit', 'cowpea_pit',
       'pigeonpea_pit', 'lentil_pit', 'oth_pulse_pit', 'soybean_pit',
       'groundnut_pit', 'coconut_pit', 'oilpalm_pit', 'sunflower_pit',
       'rapeseed_pit', 'sesameseed_pit', 'oth_oil_pit', 'sugarcane_pit',
       'sugarbeet_pit', 'cotton_pit', 'oth_fibre_pit', 'ara_coffee_pit',
       'rob_coffee_pit', 'cocoa_pit', 'tea_pit', 'tobacco_pit', 'banana_pit',
       'plantain_pit', 'trop_fruit_pit', 'temp_fruit_pit', 'vegetable_pit',
       'rest_crop_pit', 'crea_date', 'year_data', 'source', 'name_cntr',
       'name_adm1', 'name_adm2'],
      dtype='object')

In [5]:
i_df.describe()

,alloc_key,cell5m,x,y,wheat_pit,rice_pit,maize_pit,barley_pit,pearlmill_pit,smallmill_pit,...,rob_coffee_pit,cocoa_pit,tea_pit,tobacco_pit,banana_pit,plantain_pit,trop_fruit_pit,temp_fruit_pit,vegetable_pit,rest_crop_pit
count,8.328270e+05,8.328270e+05,832827.000000,832827.000000,176714.000000,132193.000000,204291.000000,77678.000000,18754.000000,14305.000000,...,7613.000000,2510.000000,1149.000000,41224.000000,21900.000000,1837.000000,67070.000000,53451.000000,250786.000000,128639.000000
mean,8.290512e+06,3.578531e+06,18.198640,20.973891,1039.630693,2694.864038,916.209012,188.585654,90.646819,65.971958,...,29.391856,23.959165,160.378749,20.392278,783.920232,438.316260,928.289109,824.017096,1793.852502,97.174339
std,3.242096e+06,1.400561e+06,73.216775,27.017823,2721.048771,5229.268160,2950.619312,551.748758,209.420726,250.657999,...,202.311540,115.512231,435.910902,129.567848,5882.757677,1523.821091,4192.006635,2990.907923,6041.513758,468.835076
min,2.352499e+06,1.013378e+06,-178.208333,-54.791667,0.050000,0.050000,0.050000,0.050000,0.100000,0.100000,...,0.100000,0.100000,0.100000,0.050000,0.100000,0.200000,0.100000,0.100000,0.100000,0.050000
25%,5.583880e+06,2.410120e+06,-51.291667,0.208333,6.800000,23.400000,9.400000,2.600000,4.300000,0.500000,...,1.100000,0.500000,2.800000,0.500000,4.200000,4.800000,8.900000,9.200000,27.100000,1.600000
50%,7.533580e+06,3.252219e+06,26.958333,27.291667,53.628284,376.543149,60.900000,18.400000,19.600000,2.700000,...,4.000000,2.450000,26.600000,2.500000,29.666667,30.200000,80.900000,72.900000,127.217021,8.500000
75%,1.078340e+07,4.656036e+06,75.625000,43.541667,505.276155,2918.700000,384.100000,115.800000,73.200000,17.200000,...,13.600000,9.300000,141.400000,11.300000,178.425000,224.600000,395.400000,422.700000,857.725000,42.400000
max,1.738134e+07,7.505176e+06,179.958333,70.458333,46983.600000,124215.251112,74495.400000,15819.322484,7607.907103,5372.170443,...,8525.800000,3074.008239,6238.000000,12209.100000,249117.200000,21608.300000,166378.309711,121546.000000,227355.200000,17072.800000


In [6]:
# We should be able to take the x,y centroid values and convert them to an equivalent 
# raster indec on our (2160, 4320) raster. Using this method, we can "stamp" the CSV
# values onto an empty raster grid.

from math import floor
from tqdm import tqdm
import numpy as np

# Function for converting from coordinates into raster indices

y_grid_cells_per_degree = 2160 / 180
x_grid_cells_per_degree = 4320 / 360

def coords_to_grid_idx(x, y):
    x = x + 180
    y = y + 90
    
    x_idx = 2160 - (y * y_grid_cells_per_degree)  # flipped due to array indexing convention
    y_idx = x * x_grid_cells_per_degree
    
    return (floor(x_idx), floor(y_idx))

# Initialise empty rasters for each crop
wheat_arr = np.zeros((2160, 4320))
rice_arr = np.zeros((2160, 4320))
maize_arr = np.zeros((2160, 4320))
barley_arr = np.zeros((2160, 4320))
pearlmill_arr = np.zeros((2160, 4320))
smallmill_arr = np.zeros((2160, 4320))
sorghum_arr = np.zeros((2160, 4320))
oth_cereal_arr = np.zeros((2160, 4320))
potato_arr = np.zeros((2160, 4320))
sweet_pot_arr = np.zeros((2160, 4320))
yams_arr = np.zeros((2160, 4320))
cassava_arr = np.zeros((2160, 4320))
oth_root_arr = np.zeros((2160, 4320))
bean_arr = np.zeros((2160, 4320))
chickpea_arr = np.zeros((2160, 4320))
cowpea_arr = np.zeros((2160, 4320))
pigeonpea_arr = np.zeros((2160, 4320))
lentil_arr = np.zeros((2160, 4320))
oth_pulse_arr = np.zeros((2160, 4320))
soybean_arr = np.zeros((2160, 4320))
groundnut_arr = np.zeros((2160, 4320))
coconut_arr = np.zeros((2160, 4320))
oilpalm_arr = np.zeros((2160, 4320))
sunflower_arr = np.zeros((2160, 4320))
rapeseed_arr = np.zeros((2160, 4320))
sesameseed_arr = np.zeros((2160, 4320))
oth_oil_arr = np.zeros((2160, 4320))
sugarcane_arr = np.zeros((2160, 4320))
sugarbeet_arr = np.zeros((2160, 4320))
cotton_arr = np.zeros((2160, 4320))
oth_fibre_arr = np.zeros((2160, 4320))
ara_coffee_arr = np.zeros((2160, 4320))
rob_coffee_arr = np.zeros((2160, 4320))
cocoa_arr = np.zeros((2160, 4320))
tea_arr = np.zeros((2160, 4320))
tobacco_arr = np.zeros((2160, 4320))
banana_arr = np.zeros((2160, 4320))
plantain_arr = np.zeros((2160, 4320))
trop_fruit_arr = np.zeros((2160, 4320))
temp_fruit_arr = np.zeros((2160, 4320))
vegetable_arr = np.zeros((2160, 4320))
rest_crop_arr = np.zeros((2160, 4320))

# Combine all our point in time production values for a given crop across
# the different systems (i, h, l, s) into individual rasters:
for idx in tqdm(range(len(i_df))):
    
    row_i = i_df.loc[idx]
    row_h = h_df.loc[idx]
    row_l = l_df.loc[idx]
    row_s = s_df.loc[idx]
    
    grid_idx = coords_to_grid_idx(row_i.x, row_i.y)
    
    for row in [row_i, row_h, row_l, row_s]:
        
        wheat_arr[grid_idx] += row.wheat_pit
        rice_arr[grid_idx] += row.rice_pit
        maize_arr[grid_idx] += row.maize_pit
        barley_arr[grid_idx] += row.barley_pit
        pearlmill_arr[grid_idx] += row.pearlmill_pit
        smallmill_arr[grid_idx] += row.smallmill_pit
        sorghum_arr[grid_idx] += row.sorghum_pit
        oth_cereal_arr[grid_idx] += row.oth_cereal_pit
        potato_arr[grid_idx] += row.potato_pit
        sweet_pot_arr[grid_idx] += row.sweet_pot_pit
        yams_arr[grid_idx] += row.yams_pit
        cassava_arr[grid_idx] += row.cassava_pit
        oth_root_arr[grid_idx] += row.oth_root_pit
        bean_arr[grid_idx] += row.bean_pit
        chickpea_arr[grid_idx] += row.chickpea_pit
        cowpea_arr[grid_idx] += row.cowpea_pit
        pigeonpea_arr[grid_idx] += row.pigeonpea_pit
        lentil_arr[grid_idx] += row.lentil_pit
        oth_pulse_arr[grid_idx] += row.oth_pulse_pit
        soybean_arr[grid_idx] += row.soybean_pit
        groundnut_arr[grid_idx] += row.groundnut_pit
        coconut_arr[grid_idx] += row.coconut_pit
        oilpalm_arr[grid_idx] += row.oilpalm_pit
        sunflower_arr[grid_idx] += row.sunflower_pit
        rapeseed_arr[grid_idx] += row.rapeseed_pit
        sesameseed_arr[grid_idx] += row.sesameseed_pit
        oth_oil_arr[grid_idx] += row.oth_oil_pit
        sugarcane_arr[grid_idx] += row.sugarcane_pit
        sugarbeet_arr[grid_idx] += row.sugarbeet_pit
        cotton_arr[grid_idx] += row.cotton_pit
        oth_fibre_arr[grid_idx] += row.oth_fibre_pit
        ara_coffee_arr[grid_idx] += row.ara_coffee_pit
        rob_coffee_arr[grid_idx] += row.rob_coffee_pit
        cocoa_arr[grid_idx] += row.cocoa_pit
        tea_arr[grid_idx] += row.tea_pit
        tobacco_arr[grid_idx] += row.tobacco_pit
        banana_arr[grid_idx] += row.banana_pit
        plantain_arr[grid_idx] += row.plantain_pit
        trop_fruit_arr[grid_idx] += row.trop_fruit_pit
        temp_fruit_arr[grid_idx] += row.temp_fruit_pit
        vegetable_arr[grid_idx] += row.vegetable_pit
        rest_crop_arr[grid_idx] += row.rest_crop_pit    

100%|██████████| 832827/832827 [19:22<00:00, 716.54it/s]


In [7]:
np.nanmax(wheat_arr)

28217.13549427897

In [8]:
np.nansum(wheat_arr)

29004810.710508406

In [9]:
wheat_arr.dtype

dtype('float64')

In [10]:
# Save the results

import rasterio

RASTER_FOR_TRANSFORM = '../../data/raw/SPAM/spam2010v2r0_global_prod.geotiff/spam2010V2r0_global_P_ACOF_A.tif'
with rasterio.open(RASTER_FOR_TRANSFORM) as transform_dataset:
    transform = transform_dataset.transform

OUTPUT_FOLDER = "../../data/processed/production_pit"

datasets_config = [
    (wheat_arr, "wheat"),
    (rice_arr, "rice"),
    (maize_arr, "maize"),
    (barley_arr, "barley"),
    (pearlmill_arr, "pearlmill"),
    (smallmill_arr, "smallmill"),
    (sorghum_arr, "sorghum"),
    (oth_cereal_arr, "oth_cereal"),
    (potato_arr, "potato"),
    (sweet_pot_arr, "sweet_pot"),
    (yams_arr, "yams"),
    (cassava_arr, "cassava"),
    (oth_root_arr, "oth_root"),
    (bean_arr, "bean"),
    (chickpea_arr, "chickpea"),
    (cowpea_arr, "cowpea"),
    (pigeonpea_arr, "pigeonpea"),
    (lentil_arr, "lentil"),
    (oth_pulse_arr, "oth_pulse"),
    (soybean_arr, "soybean"),
    (groundnut_arr, "groundnut"),
    (coconut_arr, "coconut"),
    (oilpalm_arr, "oilpalm"),
    (sunflower_arr, "sunflower"),
    (rapeseed_arr, "rapeseed"),
    (sesameseed_arr, "sesameseed"),
    (oth_oil_arr, "oth_oil"),
    (sugarcane_arr, "sugarcane"),
    (sugarbeet_arr, "sugarbeet"),
    (cotton_arr, "cotton"),
    (oth_fibre_arr, "oth_fibre"),
    (ara_coffee_arr, "ara_coffee"),
    (rob_coffee_arr, "rob_coffee"),
    (cocoa_arr, "cocoa"),
    (tea_arr, "tea"),
    (tobacco_arr, "tobacco"),
    (banana_arr, "banana"),
    (plantain_arr, "plantain"),
    (trop_fruit_arr, "trop_fruit"),
    (temp_fruit_arr, "temp_fruit"),
    (vegetable_arr, "vegetable"),
    (rest_crop_arr, "rest_crop"),
]

for config in datasets_config:
    
    arr = config[0]
    crop_name = config[1]

    dataset = rasterio.open(
        f"{OUTPUT_FOLDER}/{crop_name}_prod_pit.tif",
        "w",
        driver="GTiff",
        height=arr.shape[0],
        width=arr.shape[1],
        count=1,
        dtype=arr.dtype,
        crs='+proj=latlong',
        transform=transform,
        nodata = np.nan
    )

    dataset.write(arr, 1)
    dataset.close()




### Resample grassland data

In [11]:
# Define a re-sampling method

import rasterio
from rasterio import warp
from rasterio.enums import Resampling
import numpy as np


def resample_raster_file(src_file, resampling_method=Resampling.sum, upscale_factor=0.1):

    with rasterio.open(src_file) as dataset:
        arr = dataset.read(1)
        src_transform = dataset.transform

        # Calculate new dimensions
        new_height = int(dataset.height * upscale_factor)
        new_width = int(dataset.width * upscale_factor)
        
        # Create a destination array to receive the reprojected data
        dst_array = np.empty(shape=(new_height, new_width), dtype=arr.dtype)

        # Calculate the new transform
        dst_transform = rasterio.transform.from_bounds(*dataset.bounds, width=new_width, height=new_height)

        nodata_value = np.nan

        data, transform = warp.reproject(
            source=arr,
            destination=dst_array,
            src_transform=src_transform,
            src_crs=dataset.crs,
            dst_transform=dst_transform,
            dst_crs=dataset.crs,
            resampling=resampling_method,
            src_nodata=nodata_value,  
            dst_nodata=nodata_value,
        )

        return data, transform
    
     



BIOMASS_FROM_GRASSLAND_RASTER_FILE = f"../../data/processed/production_pit/biomass_from_grassland_tonnes.tif"

with rasterio.open(BIOMASS_FROM_GRASSLAND_RASTER_FILE) as biomass_from_grassland_dataset:
    biomass_from_grassland_arr = biomass_from_grassland_dataset.read(1)


RESAMPLED_RASTER = "../../data/processed/production_pit/biomass_from_grassland_tonnes_resampled.tif"
DOWNSCALE_FACTOR = 7200 / 4320

# Re-sample the biomass raster to crop raster (down-scale by a factor of 18)
resampled_biomass_arr, resampled_biomass_transform = resample_raster_file(
    BIOMASS_FROM_GRASSLAND_RASTER_FILE, 
    resampling_method=Resampling.sum, 
    upscale_factor=1/DOWNSCALE_FACTOR
)

# Save processed data
resampled_biomass_dataset = rasterio.open(
    RESAMPLED_RASTER,
    "w",
    driver="GTiff",
    height=int(biomass_from_grassland_arr.shape[0] / DOWNSCALE_FACTOR),
    width=int(biomass_from_grassland_arr.shape[1] / DOWNSCALE_FACTOR),
    count=1,
    dtype=resampled_biomass_arr.dtype,
    crs='+proj=latlong',
    transform=resampled_biomass_transform,
    nodata=np.nan
)
resampled_biomass_arr = np.squeeze(resampled_biomass_arr)
resampled_biomass_dataset.write(resampled_biomass_arr, 1)
resampled_biomass_dataset.close()

In [12]:
np.nansum(biomass_from_grassland_arr)

63275584536.13019

In [13]:
np.nansum(resampled_biomass_arr)

63275584536.03781

### Convert to energy and protein

In [16]:
import rasterio
import pandas as pd
import numpy as np
from tqdm import tqdm
    
OUTPUT_FOLDER = "../../data/processed"
MACRO_TABLE_PATH = "../../data/processed/lookup_tables/spam_crop_macro_lookup.csv"
macro_table = pd.read_csv(MACRO_TABLE_PATH, index_col="spam_crop_long")

RASTER_FOR_TRANSFORM = '../../data/raw/SPAM/spam2010v2r0_global_prod.geotiff/spam2010V2r0_global_P_ACOF_A.tif'
with rasterio.open(RASTER_FOR_TRANSFORM) as transform_dataset:
    transform = transform_dataset.transform


crops = ["wheat", "rice", "maize", "barley", "pearlmill", "smallmill", "sorghum", 
         "oth_cereal", "potato", "sweet_pot", "yams", "cassava", "oth_root", "bean", 
         "chickpea", "cowpea", "pigeonpea", "lentil", "oth_pulse", "soybean", "groundnut", 
         "coconut", "oilpalm", "sunflower", "rapeseed", "sesameseed", "oth_oil", 
         "sugarcane", "sugarbeet", "cotton", "oth_fibre",
         "banana", "plantain", "trop_fruit", "temp_fruit", 
         "vegetable", "rest_crop"] # exclude tobacco, tea, ara_coffee, rob_coffee, cocoa


# Let's save an array for protein and energy for each crop, and then a total protein and a
# total energy array
total_protein_arr = np.zeros((2160, 4320))
total_energy_arr = np.zeros((2160, 4320))

for crop_name in tqdm(crops):
    
    with rasterio.open(f"{OUTPUT_FOLDER}/production_pit/{crop_name}_prod_pit.tif") as dataset:
        nodata = dataset.nodata 
        arr = dataset.read(1)  

   
    crop_data = macro_table.loc[crop_name]

    print(crop_name, arr.sum())
    dry_res_arr = np.nan_to_num(arr) * crop_data.harvest_dm_fraction * crop_data.dry_rpr
    
    energy_arr = dry_res_arr * crop_data.ruminant_me * 1000 # 1000 is to convert MJ/kg to MJ/tonne
    total_energy_arr = total_energy_arr + energy_arr
    
    protein_arr = dry_res_arr * (crop_data.crude_protein / 100) # crude protein is % of mass, so divide by 100
    
    total_protein_arr = total_protein_arr + protein_arr
    


    protein_dataset = rasterio.open(
        f"{OUTPUT_FOLDER}/protein/{crop_name}_protein.tif",
        "w",
        driver="GTiff",
        height=protein_arr.shape[0],
        width=protein_arr.shape[1],
        count=1,
        dtype=protein_arr.dtype,
        crs='+proj=latlong',
        transform=transform,
        nodata=np.nan
    )

    protein_dataset.write(protein_arr, 1)
    protein_dataset.close()
    
    energy_dataset = rasterio.open(
        f"{OUTPUT_FOLDER}/energy/{crop_name}_energy.tif",
        "w",
        driver="GTiff",
        height=energy_arr.shape[0],
        width=energy_arr.shape[1],
        count=1,
        dtype=energy_arr.dtype,
        crs='+proj=latlong',
        transform=transform,
        nodata=np.nan
    )

    energy_dataset.write(energy_arr, 1)
    energy_dataset.close()

print("Done crops")    
# Now do grasses
with rasterio.open("../../data/processed/production_pit/biomass_from_grassland_tonnes_resampled.tif") as dataset:
    arr = dataset.read(1)
    
grass_data = macro_table.loc["Grassland"]

dry_res_arr = np.nan_to_num(arr) * 0.1 * grass_data.harvest_dm_fraction * grass_data.dry_rpr #included change of units (*0.1)
energy_arr = dry_res_arr * grass_data.ruminant_me * 1000 # 1000 is to convert MJ/kg to MJ/tonne
total_energy_arr = total_energy_arr + energy_arr

protein_arr = dry_res_arr * (grass_data.crude_protein / 100) # crude protein is % of mass, so divide by 100
total_protein_arr = total_protein_arr + protein_arr

protein_dataset = rasterio.open(
    f"{OUTPUT_FOLDER}/protein/grassland_protein.tif",
    "w",
    driver="GTiff",
    height=protein_arr.shape[0],
    width=protein_arr.shape[1],
    count=1,
    dtype=protein_arr.dtype,
    crs='+proj=latlong',
    transform=transform_dataset.transform,
    nodata=np.nan
)

protein_dataset.write(protein_arr, 1)
protein_dataset.close()

energy_dataset = rasterio.open(
    f"{OUTPUT_FOLDER}/energy/grassland_energy.tif",
    "w",
    driver="GTiff",
    height=energy_arr.shape[0],
    width=energy_arr.shape[1],
    count=1,
    dtype=energy_arr.dtype,
    crs='+proj=latlong',
    transform=transform_dataset.transform,
    nodata=np.nan
)
energy_dataset.write(energy_arr, 1)
energy_dataset.close()


# Write totals
total_protein_dataset = rasterio.open(
    f"{OUTPUT_FOLDER}/protein/total_protein.tif",
    "w",
    driver="GTiff",
    height=total_protein_arr.shape[0],
    width=total_protein_arr.shape[1],
    count=1,
    dtype=total_protein_arr.dtype,
    crs='+proj=latlong',
    transform=transform_dataset.transform,
    nodata=np.nan
)
total_protein_dataset.write(total_protein_arr, 1)
total_protein_dataset.close()

total_energy_dataset = rasterio.open(
    f"{OUTPUT_FOLDER}/energy/total_energy.tif",
    "w",
    driver="GTiff",
    height=total_energy_arr.shape[0],
    width=total_energy_arr.shape[1],
    count=1,
    dtype=total_energy_arr.dtype,
    crs='+proj=latlong',
    transform=transform_dataset.transform,
    nodata=np.nan
)
total_energy_dataset.write(total_energy_arr, 1)
total_energy_dataset.close()
    

  0%|          | 0/37 [00:00<?, ?it/s]

wheat nan


  3%|▎         | 1/37 [00:00<00:31,  1.14it/s]

rice nan


  5%|▌         | 2/37 [00:01<00:32,  1.07it/s]

...

vegetable nan


 97%|█████████▋| 36/37 [00:28<00:00,  1.32it/s]

rest_crop nan


100%|██████████| 37/37 [00:29<00:00,  1.24it/s]


Done crops


In [17]:
import rasterio
import numpy as np
with rasterio.open(f"../../data/processed/production_pit/wheat_prod_pit.tif") as dataset:
    arr = dataset.read(1)

In [18]:
arr = np.nan_to_num(arr, nan=0.0)
arr.max()

28217.13549427897

In [19]:
with rasterio.open("../../data/processed/production_pit/biomass_from_grassland_tonnes_resampled.tif") as dataset:
    arr = dataset.read(1)

In [20]:
np.nansum(dry_res_arr)

6327558453.603802

In [21]:
macro_table.loc["wheat"]

full_name                                        Wheat
group                                          cereals
2010_pit_prod                              592510726.8
harvest_dm_fraction                               0.88
2010_pit_prod_dry                          521409439.5
dry_rpr                                            1.0
crude_protein                                      4.2
ruminant_me                                        6.8
ndf                                               77.5
2010_pit_residues_dry (tonnes)             521409439.5
2010_pit_ruminant_me_dry (MJ)          3545584188896.0
2010_pit_crude_protein_dry (tonnes)        21899196.46
2010_pit_ruminant_ndf_dry (tonnes)         404092315.6
Name: wheat, dtype: object

In [22]:
np.nansum(total_energy_arr)

58317950189427.02

In [23]:
np.nansum(total_protein_arr)

630588815.185344